# 04 - Inundation Transition Analysis

Quantifies changes in inundation regime between pre- and post-intervention periods.

**Methods:**
- Per-pixel hydroperiod persistence (% months inundated) within OSM wetland boundary
- Inundation seasonality (coefficient of variation of monthly water presence)
- McNemar's test for change in wet pixel proportion
- Persistence class transition matrix (Rarely / Seasonally / Frequently / Persistently inundated)

**Pre/post split:** from `data/processed/breakpoint.json`

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import rasterio
import json
from scipy.stats import chi2
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import BoundaryNorm, ListedColormap
import matplotlib.patches as mpatches
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'figure.dpi': 300,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

PROCESSED_DIR = Path('data/processed')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

ANALYSIS_START = '2019-01'
ANALYSIS_END = '2024-12'

## Load Data

In [ ]:
# Load cubes
inundation_cube = xr.open_dataarray(PROCESSED_DIR / 'inundation_cube.nc')
inundation_cube = inundation_cube.sel(time=slice(ANALYSIS_START, ANALYSIS_END))

# Load wetland mask
with rasterio.open('data/raw/wetland_mask.tif') as src:
    mask_arr = src.read(1)
    mask_transform = src.transform
    mask_crs = src.crs

wetland_mask = xr.DataArray(mask_arr == 1, dims=['y', 'x'],
                             coords={'y': inundation_cube.y, 'x': inundation_cube.x})

# Load breakpoint
with open(PROCESSED_DIR / 'breakpoint.json') as f:
    bp_data = json.load(f)

breakpoint_date = pd.Timestamp(bp_data['primary_breakpoint'])
print(f"Primary breakpoint: {breakpoint_date.strftime('%Y-%m')}")

# Split into pre/post periods
pre_cube = inundation_cube.sel(time=inundation_cube.time < np.datetime64(breakpoint_date))
post_cube = inundation_cube.sel(time=inundation_cube.time >= np.datetime64(breakpoint_date))

print(f"Pre period:  {pre_cube.time.values[0]} to {pre_cube.time.values[-1]} ({len(pre_cube.time)} months)")
print(f"Post period: {post_cube.time.values[0]} to {post_cube.time.values[-1]} ({len(post_cube.time)} months)")

## Per-Pixel Hydroperiod Persistence

In [ ]:
def compute_persistence(cube, mask):
    """
    Compute per-pixel hydroperiod persistence (fraction of months inundated)
    within the wetland mask. Returns DataArray of values 0-1, NaN outside mask.
    """
    valid = (cube >= 0) & mask  # valid data within wetland
    water = (cube == 1) & mask
    n_valid = valid.sum(dim='time').astype(float)
    n_water = water.sum(dim='time').astype(float)
    persistence = xr.where(n_valid > 0, n_water / n_valid, np.nan)
    persistence = persistence.where(mask)
    return persistence

pre_persist = compute_persistence(pre_cube, wetland_mask)
post_persist = compute_persistence(post_cube, wetland_mask)
diff_persist = post_persist - pre_persist

print(f"Pre  persistence: mean={float(pre_persist.mean(skipna=True)):.3f}, "
      f"std={float(pre_persist.std(skipna=True)):.3f}")
print(f"Post persistence: mean={float(post_persist.mean(skipna=True)):.3f}, "
      f"std={float(post_persist.std(skipna=True)):.3f}")
print(f"Diff (post-pre):  mean={float(diff_persist.mean(skipna=True)):.3f}")

In [ ]:
def compute_cv(cube, mask):
    """Coefficient of variation of monthly inundation within mask."""
    water = (cube == 1).astype(float).where(mask)
    mean_w = water.mean(dim='time')
    std_w = water.std(dim='time')
    cv = xr.where(mean_w > 0, std_w / mean_w, np.nan)
    return cv.where(mask)

pre_cv = compute_cv(pre_cube, wetland_mask)
post_cv = compute_cv(post_cube, wetland_mask)

print(f"Pre  CV: mean={float(pre_cv.mean(skipna=True)):.3f}")
print(f"Post CV: mean={float(post_cv.mean(skipna=True)):.3f}")

## Figure 1 — Hydroperiod Persistence Maps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5.5))

cmap_persist = plt.cm.YlOrRd
cmap_diff = plt.cm.RdBu_r

wetland_vals = wetland_mask.values.astype(float)
wetland_vals[~wetland_mask.values] = np.nan

def get_extent(da):
    """Return imshow extent [xmin, xmax, ymin, ymax]."""
    x = da.x.values
    y = da.y.values
    dx = abs(x[1] - x[0]) / 2
    dy = abs(y[1] - y[0]) / 2
    return [x[0] - dx, x[-1] + dx, y[-1] - dy, y[0] + dy]

ext = get_extent(pre_persist)

im0 = axes[0].imshow(pre_persist.values, cmap=cmap_persist, vmin=0, vmax=1,
                      extent=ext, origin='upper', aspect='equal')
axes[0].set_title(f'Pre-intervention\n({pd.Timestamp(pre_cube.time.values[0]).strftime("%Y-%m")} \u2013 '
                  f'{pd.Timestamp(pre_cube.time.values[-1]).strftime("%Y-%m")})')
plt.colorbar(im0, ax=axes[0], label='Persistence (fraction)', shrink=0.8)

im1 = axes[1].imshow(post_persist.values, cmap=cmap_persist, vmin=0, vmax=1,
                      extent=ext, origin='upper', aspect='equal')
axes[1].set_title(f'Post-intervention\n({pd.Timestamp(post_cube.time.values[0]).strftime("%Y-%m")} \u2013 '
                  f'{pd.Timestamp(post_cube.time.values[-1]).strftime("%Y-%m")})')
plt.colorbar(im1, ax=axes[1], label='Persistence (fraction)', shrink=0.8)

max_diff = max(abs(float(diff_persist.min(skipna=True))), abs(float(diff_persist.max(skipna=True))))
im2 = axes[2].imshow(diff_persist.values, cmap=cmap_diff, vmin=-max_diff, vmax=max_diff,
                      extent=ext, origin='upper', aspect='equal')
axes[2].set_title('Change (Post \u2212 Pre)')
plt.colorbar(im2, ax=axes[2], label='\u0394 Persistence (fraction)', shrink=0.8)

for ax in axes:
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    ax.ticklabel_format(style='sci', axis='both', scilimits=(0,0))

plt.suptitle('Hydroperiod Persistence \u2014 Wiang Nong Lom Wetland', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_04_persistence_maps.pdf', bbox_inches='tight')
plt.savefig(OUTPUT_DIR / 'fig_04_persistence_maps.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: outputs/fig_04_persistence_maps.pdf/.png")

## McNemar's Test for Change in Wet Pixel Proportion

In [ ]:
# Binary classification: wet = inundated in majority of months (persistence > 0.5)
pre_wet = (pre_persist > 0.5).values
post_wet = (post_persist > 0.5).values
in_mask = wetland_mask.values

# Contingency table (only wetland pixels with valid data)
valid_px = in_mask & ~np.isnan(pre_persist.values) & ~np.isnan(post_persist.values)

pre_w = pre_wet[valid_px]
post_w = post_wet[valid_px]

# McNemar's table: rows = pre, cols = post
n_00 = ((~pre_w) & (~post_w)).sum()   # dry -> dry
n_01 = ((~pre_w) & post_w).sum()      # dry -> wet
n_10 = (pre_w & (~post_w)).sum()      # wet -> dry
n_11 = (pre_w & post_w).sum()         # wet -> wet

print("McNemar Contingency Table:")
print(f"                Post-Dry   Post-Wet")
print(f"Pre-Dry:        {n_00:8,}   {n_01:8,}")
print(f"Pre-Wet:        {n_10:8,}   {n_11:8,}")

# McNemar statistic with continuity correction
b, c = n_01, n_10
if (b + c) > 0:
    chi2_stat = (abs(b - c) - 1)**2 / (b + c)
    p_value = 1 - chi2.cdf(chi2_stat, df=1)
    print(f"\nMcNemar \u03c7\u00b2 (with continuity correction) = {chi2_stat:.4f}")
    print(f"p-value = {p_value:.4e}")
    print(f"Conclusion: {'Significant change (p < 0.05)' if p_value < 0.05 else 'No significant change'} in wet pixel proportion")
else:
    print("No discordant pairs \u2014 cannot compute McNemar statistic.")

## Persistence Class Transition Matrix

In [ ]:
# Classify pixels into persistence regimes
def classify_persistence(persist_da):
    """
    0: Rarely     < 20%
    1: Seasonally 20-60%
    2: Frequently 60-90%
    3: Persistently > 90%
    """
    p = persist_da.values
    cls = np.full(p.shape, -1, dtype=np.int8)
    cls[p < 0.20] = 0
    cls[(p >= 0.20) & (p < 0.60)] = 1
    cls[(p >= 0.60) & (p < 0.90)] = 2
    cls[p >= 0.90] = 3
    cls[np.isnan(p)] = -1
    return cls

pre_class = classify_persistence(pre_persist)
post_class = classify_persistence(post_persist)

class_labels = ['Rarely\n(<20%)', 'Seasonally\n(20\u201360%)', 'Frequently\n(60\u201390%)', 'Persistently\n(>90%)']
n_classes = 4

# Build transition matrix (count pixels)
valid_px2 = wetland_mask.values & (pre_class >= 0) & (post_class >= 0)
trans_matrix = np.zeros((n_classes, n_classes), dtype=int)
for i in range(n_classes):
    for j in range(n_classes):
        trans_matrix[i, j] = ((pre_class == i) & (post_class == j) & valid_px2).sum()

# Convert to percentages of total valid wetland pixels
total_px = valid_px2.sum()
trans_pct = trans_matrix / total_px * 100

print("Transition matrix (% of total wetland pixels):")
df_trans = pd.DataFrame(trans_pct, index=[f'Pre: {l.replace(chr(10)," ")}' for l in class_labels],
                         columns=[f'Post: {l.replace(chr(10)," ")}' for l in class_labels])
print(df_trans.round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# --- Left: transition matrix heatmap ---
ax = axes[0]
im = ax.imshow(trans_pct, cmap='Blues', vmin=0, aspect='auto')

for i in range(n_classes):
    for j in range(n_classes):
        val = trans_pct[i, j]
        txt_color = 'white' if val > trans_pct.max() * 0.5 else '#222'
        ax.text(j, i, f'{val:.1f}%', ha='center', va='center', fontsize=9, color=txt_color)

ax.set_xticks(range(n_classes))
ax.set_yticks(range(n_classes))
ax.set_xticklabels(class_labels, fontsize=8)
ax.set_yticklabels(class_labels, fontsize=8)
ax.set_xlabel('Post-intervention Regime')
ax.set_ylabel('Pre-intervention Regime')
ax.set_title('Persistence Class Transition Matrix\n(% of wetland pixels)')
plt.colorbar(im, ax=ax, label='% of pixels', shrink=0.8)

# Diagonal indicator
for k in range(n_classes):
    ax.add_patch(plt.Rectangle((k - 0.5, k - 0.5), 1, 1, fill=False,
                                edgecolor='#d62728', linewidth=2))

# --- Right: stacked bar showing pre/post class distributions ---
ax2 = axes[1]
pre_counts = np.array([((pre_class == i) & wetland_mask.values).sum() for i in range(n_classes)])
post_counts = np.array([((post_class == i) & wetland_mask.values).sum() for i in range(n_classes)])
pre_pct_bar = pre_counts / pre_counts.sum() * 100
post_pct_bar = post_counts / post_counts.sum() * 100

bar_colors = ['#fee8c8', '#fdbb84', '#e34a33', '#7f0000']
x = np.arange(n_classes)
width = 0.35
for i in range(n_classes):
    ax2.bar(i - width/2, pre_pct_bar[i], width, color=bar_colors[i],
            edgecolor='white', linewidth=0.5, label=f'Pre {class_labels[i]}' if i == 0 else None)
    ax2.bar(i + width/2, post_pct_bar[i], width, color=bar_colors[i],
            edgecolor='black', linewidth=0.8, alpha=0.85)

# Legend
from matplotlib.patches import Patch as MPatch
legend_els = [MPatch(facecolor=bar_colors[i], label=class_labels[i].replace('\n', ' '))
              for i in range(n_classes)]
legend_els += [MPatch(facecolor='grey', alpha=0.4, label='Pre (solid)'),
               MPatch(facecolor='grey', label='Post (hatched border)')]

ax2.set_xticks(x)
ax2.set_xticklabels(class_labels, fontsize=8)
ax2.set_ylabel('% of wetland pixels')
ax2.set_title('Persistence Class Distribution\nPre vs Post')
ax2.legend(handles=legend_els, fontsize=7.5, loc='upper right', ncol=2)
ax2.grid(True, alpha=0.25, axis='y')

# Annotate change arrows
for i in range(n_classes):
    delta = post_pct_bar[i] - pre_pct_bar[i]
    max_bar = max(pre_pct_bar[i], post_pct_bar[i])
    sign = '+' if delta >= 0 else ''
    ax2.text(i, max_bar + 1, f'{sign}{delta:.1f}pp', ha='center', fontsize=8,
             color='#d62728' if delta > 0 else '#2166ac')

plt.suptitle('Inundation Regime Transitions \u2014 Wiang Nong Lom Wetland', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_05_transition_matrix.pdf', bbox_inches='tight')
plt.savefig(OUTPUT_DIR / 'fig_05_transition_matrix.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: outputs/fig_05_transition_matrix.pdf/.png")

## Figure 3 — Inundation Seasonality (CV)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ext = get_extent(pre_cv)

for ax, cv_da, title in [
    (axes[0], pre_cv, f'Pre-intervention CV'),
    (axes[1], post_cv, f'Post-intervention CV'),
]:
    im = ax.imshow(cv_da.values, cmap='plasma_r', vmin=0, vmax=float(max(pre_cv.max(skipna=True), post_cv.max(skipna=True))),
                   extent=ext, origin='upper', aspect='equal')
    ax.set_title(title)
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    ax.ticklabel_format(style='sci', axis='both', scilimits=(0,0))
    plt.colorbar(im, ax=ax, label='CV of monthly inundation', shrink=0.8)

plt.suptitle('Inundation Seasonality (Coefficient of Variation)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_06_cv_maps.pdf', bbox_inches='tight')
plt.savefig(OUTPUT_DIR / 'fig_06_cv_maps.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: outputs/fig_06_cv_maps.pdf/.png")

In [ ]:
# Save persistence arrays for downstream notebooks
pre_persist.to_netcdf(PROCESSED_DIR / 'pre_persistence.nc')
post_persist.to_netcdf(PROCESSED_DIR / 'post_persistence.nc')
print("Saved: data/processed/pre_persistence.nc, post_persistence.nc")

## Summary

| Metric | Pre | Post | Change |
|--------|-----|------|--------|
| Mean persistence (wetland) | (see above) | (see above) | (see above) |
| Wet pixel proportion | (see McNemar table) | (see McNemar table) | (see McNemar p-value) |

**Next:** `05_vegetation_response.ipynb`